# 05 · Gradient geometry and AIM interventionThree heatmaps from one measurement cache:| | ||---|---|| **cosine similarity** | does the backbone change task-gradient geometry? || **conflict frequency** | which task pairs conflict, and how often? || **w_ij** | does AIM intervene differently on a pretrained backbone? |Needs the cache written by `10_measure_gradients.ipynb`. Use the **LS**checkpoint for geometry (it isolates the backbone — AIM's geometry is partly itsown doing) and the **AIM-Matrix** checkpoint for `w_ij` (τ and cos must comefrom the same model state).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, ".")
import numpy as np, matplotlib.pyplot as plt
import common as C
C.apply_style()
print("results trees:")
for b, d in C.RESULTS.items():
    for tag, p in d.items():
        print(f"  {b:9s} {tag:7s} {'OK ' if p.is_dir() else 'MISSING'} {p}")

In [ ]:
TAG = "11task"
METHOD = "ls"            # "ls" for geometry, "aim_matrix" for w_ij
KIND = "cos"             # "cos" | "freq" | "weight"
K = 10.0                 # AIM gate temperature; must match training
ANNOTATE = False         # values in every cell — busy at 11 tasks

cache = C.load_cache(TAG, METHOD)
tasks = cache[list(cache)[0]]["tasks"]
for b, e in cache.items():
    off = ~np.eye(len(tasks), dtype=bool)
    print(f"{b:9s} {e['cos'].shape[0]} batches   mean off-diag cos "
          f"{e['cos'].mean(0)[off].mean():+.4f}   "
          f"{100*(e['cos'][:, off] < 0).mean():.1f}% conflicting")

In [ ]:
def panels(cache, kind, k=10.0):
    out = {}
    for b, e in cache.items():
        if kind == "cos":
            out[b] = e["cos"].mean(0)
        elif kind == "freq":
            out[b] = (e["cos"] < 0).mean(0)
        else:
            if e["tau"] is None:
                raise ValueError(f"{b}: no tau in this cache — use METHOD='aim_matrix'")
            out[b] = C.aim_weight(e["cos"], e["tau"], k).mean(0)
    return out

SPEC = {
  "cos":    (C.cmap("diverging"),  None, "{:+.2f}",
             "conflict  <-   mean cos phi_ij   ->  aligned",
             "Average gradient cosine similarity: does the backbone change task-gradient geometry?"),
  "freq":   (C.cmap("sequential"), (0, 1), "{:.2f}",
             "fraction of batches with cos phi_ij < 0",
             "Gradient conflict frequency: which task pairs conflict, and how often?"),
  "weight": (C.cmap("sequential"), (0, 1), "{:.2f}",
             "mean w_ij   0 = leave alone   ->   1 = remove fully",
             "AIM intervention weight: does AIM intervene differently on a pretrained backbone?"),
}

mats = panels(cache, KIND, K)
order = [b for b in ("Uni-Mol", "GNN") if b in mats]
cm, lim, fmt, cb_label, head = SPEC[KIND]
off = ~np.eye(len(tasks), dtype=bool)
if lim is None:
    v = max(abs(M[off]).max() for M in mats.values()); vmin, vmax = -v, v
else:
    vmin, vmax = lim

pair = len(order) == 2
widths = [1] * len(order) + [.05] + ([1, .05] if pair else [])
fig = plt.figure(figsize=(5.4 * (len(order) + (1 if pair else 0)), 5.6))
gs = fig.add_gridspec(1, len(widths), width_ratios=widths, wspace=.34,
                      left=.06, right=.95, top=.76, bottom=.17)
ims = [C.heatmap(fig.add_subplot(gs[0, i]), mats[b], tasks, cm, vmin, vmax, b,
                 fmt, ANNOTATE) for i, b in enumerate(order)]
cb = fig.colorbar(ims[0], cax=fig.add_subplot(gs[0, len(order)]))
cb.outline.set_visible(False); cb.ax.tick_params(length=0, colors=C.MUTED, labelsize=8)
cb.ax.yaxis.set_label_position("left"); cb.set_label(cb_label, color=C.SECONDARY, fontsize=8.5)

if pair:
    diff = mats[order[0]] - mats[order[1]]
    d = abs(diff[off]).max() or .05
    ax = fig.add_subplot(gs[0, len(order) + 1])
    im = C.heatmap(ax, diff, tasks, C.cmap("diverging"), -d, d,
                   f"{order[0]} - {order[1]}", "{:+.2f}", ANNOTATE)
    cb2 = fig.colorbar(im, cax=fig.add_subplot(gs[0, len(order) + 2]))
    cb2.outline.set_visible(False); cb2.ax.tick_params(length=0, colors=C.MUTED, labelsize=8)
    cb2.set_label("difference", color=C.SECONDARY, fontsize=8.5)

fig.suptitle(head, x=.01, y=.975, ha="left", fontsize=15, fontweight="bold", color=C.INK)
fig.text(.01, .895, f"{len(tasks)} QM9 tasks · {METHOD.upper()} checkpoint, primary split · "
         f"{cache[order[0]]['cos'].shape[0]} batches · n_train={C.N_TRAIN}, seed {C.SEED}",
         fontsize=9.5, color=C.MUTED, ha="left")
fig.text(.01, .03, "Diagonal masked (i = j carries no information). Measured — "
         "train.py never logs the per-step cosines.", fontsize=7.5, color=C.MUTED)
C.save(fig, f"{KIND}_{TAG}_{METHOD}", TAG); plt.show()

### All three at once

In [ ]:
for method, kinds in (("ls", ["cos", "freq"]), ("aim_matrix", ["weight"])):
    try:
        cache = C.load_cache(TAG, method)
    except FileNotFoundError as e:
        print("skip:", e); continue
    for kind in kinds:
        print(f"  {method} / {kind}")   # re-run the cell above with these set